# PatchCore Inference Notebook
**Backbone**: WideResNet50-2 (ImageNet pretrained) | **Reference images**: same 2 as FADE

Run cells 1–5 once (imports, config, model + memory bank, helpers). Then run one cell per file to get the anomaly score, plots (if `PLOT_HEATMAPS` is on), and pixel/instance precision, recall, and F1.

In [ ]:
import cv2, h5py, numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
import torch
import torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms as T
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.metrics import precision_recall_curve as _prc

Imports OK


In [ ]:
BASE_DIR = Path('PLACEHOLDER')
REF_PATHS = [BASE_DIR / 'image.png1',
             BASE_DIR / 'image2.png']

IMG_SIZE                = 224   # WideResNet50-2 input resolution
EVAL_IMG_SIZE           = 448   # evaluation / plot resolution (matches FADE)
NORMALIZE_SEGMENTATIONS = False
PLOT_HEATMAPS           = False  # set False to skip all plotting and just print metrics
SAVE_DIR = Path('output_patchcore')
SAVE_DIR.mkdir(exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

Device: cpu


In [18]:
# --- Load WideResNet50-2 and register feature hooks (run once) ---
backbone = models.wide_resnet50_2(weights=models.Wide_ResNet50_2_Weights.IMAGENET1K_V1)
backbone.eval().to(device)

_feats = {}
def _hook_l2(m, i, o): _feats['layer2'] = o.detach()
def _hook_l3(m, i, o): _feats['layer3'] = o.detach()
backbone.layer2.register_forward_hook(_hook_l2)
backbone.layer3.register_forward_hook(_hook_l3)

_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE), interpolation=T.InterpolationMode.BICUBIC),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def _extract(pil_img):
    """Return (patches [H*W, 1536], H, W) for a PIL image."""
    x = _transform(pil_img).unsqueeze(0).to(device)
    with torch.no_grad():
        backbone(x)
    l2 = _feats['layer2']                                              # [1, 512, H2, W2]
    l3 = _feats['layer3']                                              # [1,1024, H3, W3]
    H, W = l2.shape[2], l2.shape[3]
    l3_up = F.interpolate(l3, size=(H, W), mode='bilinear', align_corners=False)
    feat = torch.cat([l2, l3_up], dim=1)                              # [1,1536, H, W]
    patches = feat[0].permute(1, 2, 0).reshape(-1, 1536).cpu().numpy()
    return patches, H, W

# Build memory bank from reference images
_mb = []
for rp in REF_PATHS:
    patches, _, _ = _extract(Image.open(rp).convert('RGB'))
    _mb.append(patches)
memory_bank = np.concatenate(_mb, axis=0)                             # [N_patches, 1536]
memory_bank_t = torch.tensor(memory_bank, dtype=torch.float32, device=device)
print(f'Memory bank: {memory_bank_t.shape}  ({len(REF_PATHS)} reference images, {memory_bank_t.shape[0]} patches)')

Memory bank: torch.Size([1568, 1536])  (2 reference images, 1568 patches)


In [19]:
# --- Core PatchCore helpers ---

def predict_anomaly_map(pil_image):
    """Return (score_map [EVAL_IMG_SIZE, EVAL_IMG_SIZE], image_score=max)."""
    patches, H, W = _extract(pil_image)
    q    = torch.tensor(patches, dtype=torch.float32, device=device)
    q_sq = (q ** 2).sum(1, keepdim=True)
    m_sq = (memory_bank_t ** 2).sum(1, keepdim=True)
    d2   = (q_sq + m_sq.T - 2.0 * (q @ memory_bank_t.T)).clamp(min=0.0)
    min_d = d2.min(1).values.sqrt().cpu().numpy()
    score_map = cv2.resize(
        min_d.reshape(H, W).astype(np.float32),
        (EVAL_IMG_SIZE, EVAL_IMG_SIZE),
        interpolation=cv2.INTER_LINEAR,
    )
    return score_map, float(score_map.max())

def _apply_size_filter(score_map, thr, min_sz, max_sz):
    bm = (score_map > thr).astype(np.uint8)
    if not bm.any():
        return bm
    n, lbl, st, _ = cv2.connectedComponentsWithStats(bm, connectivity=8)
    out = np.zeros_like(bm)
    for i in range(1, n):
        a = st[i, cv2.CC_STAT_AREA]
        if (min_sz == 0 or a >= min_sz) and (max_sz == 0 or a <= max_sz):
            out[lbl == i] = 1
    return out

def auto_threshold(score_map, gt_mask=None, percentile=95, ignore_mask=None):
    """Return (thr_gt, thr_pct): F1-optimal threshold from GT (None if no GT)
    and a percentile-based threshold (always available, used as fallback)."""
    thr_pct = float(np.percentile(score_map, percentile))
    thr_gt = None
    if gt_mask is not None:
        gt_r = cv2.resize(gt_mask.astype(np.uint8),
                          (score_map.shape[1], score_map.shape[0]),
                          interpolation=cv2.INTER_NEAREST)
        gt_b = (gt_r > 0).astype(np.uint8)
        if ignore_mask is not None:
            valid = cv2.resize(ignore_mask.astype(np.uint8),
                               (score_map.shape[1], score_map.shape[0]),
                               interpolation=cv2.INTER_NEAREST) == 0
        else:
            valid = np.ones(score_map.shape, dtype=bool)
        sv, lv = score_map[valid].ravel(), gt_b[valid].ravel()
        if lv.sum() > 0 and (len(lv) - lv.sum()) > 0:
            prec, rec, thrs = _prc(lv, sv)
            f1s  = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-8)
            thr_gt = float(thrs[int(np.argmax(f1s))])
    return thr_gt, thr_pct

print('Helper functions ready.')

# --- Instance-level evaluation helpers ---
# Matching: centroid-hit (one-to-many). A GT instance is a TP if any pred
# blob centroid falls within CENTROID_MATCH_RADIUS pixels of that GT
# instance's centroid.
CENTROID_MATCH_RADIUS = 50   # px at EVAL_IMG_SIZE resolution (~3x patch size)

def _extract_instances(binary_mask):
    if not binary_mask.any():
        return []
    n, labels, stats, centroids = cv2.connectedComponentsWithStats(
        binary_mask.astype(np.uint8), connectivity=8)
    return [{
        'id': i,
        'mask': (labels == i),
        'area': int(stats[i, cv2.CC_STAT_AREA]),
        'centroid': (float(centroids[i, 0]), float(centroids[i, 1])),
    } for i in range(1, n)]

def _match_instances(gt_instances, pred_instances, radius=CENTROID_MATCH_RADIUS):
    gt_matched, pred_matched = set(), set()
    for gi, gt_inst in enumerate(gt_instances):
        gx, gy = gt_inst['centroid']
        for pi, pred_inst in enumerate(pred_instances):
            px, py = pred_inst['centroid']
            if ((px - gx) ** 2 + (py - gy) ** 2) ** 0.5 <= radius:
                gt_matched.add(gi)
                pred_matched.add(pi)
    tp = len(gt_matched)
    fn = len(gt_instances) - tp
    fp = len(pred_instances) - len(pred_matched)
    return tp, fp, fn, gt_matched, pred_matched

def instance_eval(score_map, gt_mask, threshold, eval_size,
                  ignore_mask=None, min_sz=0, max_sz=0,
                  radius=CENTROID_MATCH_RADIUS):
    """Compute instance-level P/R/F1 at a single threshold."""
    gt_r = cv2.resize(gt_mask.astype(np.uint8),
                      (eval_size, eval_size), interpolation=cv2.INTER_NEAREST)
    gt_b = (gt_r > 0).astype(np.uint8)

    if ignore_mask is not None:
        ign_r = cv2.resize(ignore_mask.astype(np.uint8),
                           (eval_size, eval_size), interpolation=cv2.INTER_NEAREST)
        valid = (ign_r == 0)
        gt_b[~valid] = 0
        sm = score_map.copy()
        sm[~valid] = 0.0
    else:
        sm = score_map

    gt_instances = _extract_instances(gt_b)
    pred_binary = _apply_size_filter(sm, threshold, min_sz, max_sz)
    pred_instances = _extract_instances(pred_binary)

    tp, fp, fn, gt_m, pred_m = _match_instances(gt_instances, pred_instances, radius=radius)
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
    return {
        'threshold': threshold,
        'n_gt': len(gt_instances), 'n_pred': len(pred_instances),
        'tp': tp, 'fp': fp, 'fn': fn,
        'precision': prec, 'recall': rec, 'f1': f1,
        'gt_instances': gt_instances, 'pred_instances': pred_instances,
        'gt_matched': gt_m, 'pred_matched': pred_m,
    }

print('Instance-level helpers ready.')

Helper functions ready.
Instance-level helpers ready.


In [20]:
# --- Plotting (only runs when PLOT_HEATMAPS is True) ---

def _plot_overview(orig_rgb, score_map, gt_mask, pred_mask, threshold, stem):
    overlay = orig_rgb.copy()
    pm = pred_mask.astype(bool)
    overlay[pm] = (orig_rgb[pm] * 0.4 + np.array([255, 0, 0]) * 0.6).astype(np.uint8)

    n_panels = 3 + (gt_mask is not None)
    fig, axes = plt.subplots(1, n_panels, figsize=(5 * n_panels, 5))

    axes[0].imshow(orig_rgb); axes[0].set_title('Original'); axes[0].axis('off')
    axes[1].imshow(overlay); axes[1].set_title(f'Overlay (thr={threshold:.3f})'); axes[1].axis('off')
    im = axes[2].imshow(score_map, cmap='hot'); axes[2].set_title('Anomaly heatmap'); axes[2].axis('off')
    plt.colorbar(im, ax=axes[2], fraction=0.046)
    if gt_mask is not None:
        gt_vis = cv2.resize((gt_mask > 0).astype(np.uint8),
                            (EVAL_IMG_SIZE, EVAL_IMG_SIZE), interpolation=cv2.INTER_NEAREST)
        axes[3].imshow(gt_vis, cmap='gray'); axes[3].set_title('Ground truth'); axes[3].axis('off')

    plt.suptitle(stem)
    plt.tight_layout()
    fig.savefig(SAVE_DIR / f'{stem}_overview.png', bbox_inches='tight', dpi=100)
    plt.show()

def _plot_instances(orig_rgb, inst, stem):
    overlay = orig_rgb.copy()
    for pi, pred in enumerate(inst['pred_instances']):
        if pi not in inst['pred_matched']:
            overlay[pred['mask']] = (orig_rgb[pred['mask']] * 0.4 + np.array([255, 140, 0]) * 0.6).astype(np.uint8)
    for gi, gt in enumerate(inst['gt_instances']):
        color = np.array([0, 255, 0]) if gi in inst['gt_matched'] else np.array([255, 0, 0])
        dil = cv2.dilate(gt['mask'].astype(np.uint8), np.ones((9, 9), np.uint8)).astype(bool)
        overlay[dil] = (orig_rgb[dil] * 0.25 + color * 0.75).astype(np.uint8)

    fig, ax = plt.subplots(figsize=(7, 7))
    ax.imshow(overlay)
    ax.set_title(f'Instances — TP(green)={inst["tp"]}  FP(orange)={inst["fp"]}  FN(red)={inst["fn"]}')
    ax.axis('off')
    plt.suptitle(f'{stem} — instances')
    plt.tight_layout()
    fig.savefig(SAVE_DIR / f'{stem}_instances.png', bbox_inches='tight', dpi=100)
    plt.show()

print('Plotting helpers ready.')

Plotting helpers ready.


In [21]:
# --- Single entry point: load a file, run inference, print metrics, plot if enabled ---

def evaluate_file(label, path, percentile=95, min_sz=0, max_sz=0):
    stem = Path(path).stem
    with h5py.File(path, 'r') as f:
        img = f['img'][:]
        gt_mask = f['mask'][:] if 'mask' in f else None
        ignore_mask = f['ignore_mask'][:] if 'ignore_mask' in f else None

    pil = Image.fromarray(img).convert('RGB')
    orig_rgb = np.array(pil.resize((EVAL_IMG_SIZE, EVAL_IMG_SIZE), Image.BICUBIC))

    score_map, _ = predict_anomaly_map(pil)
    if NORMALIZE_SEGMENTATIONS:
        lo, hi = score_map.min(), score_map.max()
        score_map = (score_map - lo) / (hi - lo + 1e-8)

    thr_gt, thr_pct = auto_threshold(score_map, gt_mask=gt_mask,
                                     percentile=percentile, ignore_mask=ignore_mask)
    threshold = thr_gt if thr_gt is not None else thr_pct
    pred_mask = _apply_size_filter(score_map, threshold, min_sz, max_sz)

    result = {'label': label, 'stem': stem, 'threshold': threshold}
    print(f'{label} ({stem})  threshold={threshold:.4f}')

    if gt_mask is not None:
        gt_r = cv2.resize(gt_mask.astype(np.uint8), (EVAL_IMG_SIZE, EVAL_IMG_SIZE),
                          interpolation=cv2.INTER_NEAREST)
        gt_b = (gt_r > 0).astype(np.uint8)
        valid = (cv2.resize(ignore_mask.astype(np.uint8), (EVAL_IMG_SIZE, EVAL_IMG_SIZE),
                            interpolation=cv2.INTER_NEAREST) == 0) if ignore_mask is not None \
                else np.ones_like(gt_b, dtype=bool)
        gt_f, pred_f = gt_b[valid], pred_mask[valid]

        pixel = {
            'precision': precision_score(gt_f, pred_f, zero_division=0),
            'recall':    recall_score(gt_f, pred_f, zero_division=0),
            'f1':        f1_score(gt_f, pred_f, zero_division=0),
        }
        inst = instance_eval(score_map, gt_mask, threshold, EVAL_IMG_SIZE,
                             ignore_mask=ignore_mask, min_sz=min_sz, max_sz=max_sz)
        result['pixel'] = pixel
        result['instance'] = {'precision': inst['precision'], 'recall': inst['recall'], 'f1': inst['f1']}

        print(f'  Pixel    — P={pixel["precision"]:.4f}  R={pixel["recall"]:.4f}  F1={pixel["f1"]:.4f}')
        print(f'  Instance — P={inst["precision"]:.4f}  R={inst["recall"]:.4f}  F1={inst["f1"]:.4f}'
              f'  (GT={inst["n_gt"]}, TP={inst["tp"]}, FP={inst["fp"]}, FN={inst["fn"]})')

        if PLOT_HEATMAPS:
            _plot_overview(orig_rgb, score_map, gt_mask, pred_mask, threshold, stem)
            _plot_instances(orig_rgb, inst, stem)
    else:
        print('  No ground truth — skipping metrics.')
        if PLOT_HEATMAPS:
            _plot_overview(orig_rgb, score_map, gt_mask, pred_mask, threshold, stem)

    return result

results = {}
print('Evaluation function ready.')

Evaluation function ready.


In [22]:
results['File1'] = evaluate_file(
    'File1', 'train_data/20251211_FM_Database_Die_exp8_red19_1.hdf5')
results['File2'] = evaluate_file(
    'File2', 'train_data/20251211_FM_Database_Die_exp8_red19_7.hdf5')
results['File3'] = evaluate_file(
    'File3', 'train_data/20251211_FM_Database_Die_exp8_red19_8.hdf5')
results['File4'] = evaluate_file(
    'File4', 'train_data/20251211_FM_Database_Die_exp8_yellow12_7.hdf5')
results['File5'] = evaluate_file(
    'File5', 'test_data/20251211_FM_Database_Die_exp8_red19_14.hdf5')

File1 (20251211_FM_Database_Die_exp8_red19_1)  threshold=5.0322
  Pixel    — P=0.2132  R=0.5046  F1=0.2997
  Instance — P=1.0000  R=0.1538  F1=0.2667  (GT=13, TP=2, FP=0, FN=11)
File2 (20251211_FM_Database_Die_exp8_red19_7)  threshold=4.4992
  Pixel    — P=0.0294  R=0.3133  F1=0.0537
  Instance — P=0.8333  R=0.1852  F1=0.3030  (GT=27, TP=5, FP=1, FN=22)
File3 (20251211_FM_Database_Die_exp8_red19_8)  threshold=5.0146
  Pixel    — P=0.0531  R=0.4424  F1=0.0949
  Instance — P=0.8571  R=0.3333  F1=0.4800  (GT=18, TP=6, FP=1, FN=12)
File4 (20251211_FM_Database_Die_exp8_yellow12_7)  threshold=4.5076
  Pixel    — P=0.0956  R=0.3636  F1=0.1514
  Instance — P=1.0000  R=0.2000  F1=0.3333  (GT=15, TP=3, FP=0, FN=12)
File5 (20251211_FM_Database_Die_exp8_red19_14)  threshold=3.1863
  Pixel    — P=0.0037  R=0.5333  F1=0.0073
  Instance — P=0.4000  R=0.6667  F1=0.5000  (GT=3, TP=2, FP=3, FN=1)


In [23]:
print('=== Summary (pixel / instance) ===')
for name, r in results.items():
    if 'pixel' in r:
        print(f"{name:8s}  pixel F1={r['pixel']['f1']:.4f}  P={r['pixel']['precision']:.4f}  R={r['pixel']['recall']:.4f}"
              f"   |   instance F1={r['instance']['f1']:.4f}  P={r['instance']['precision']:.4f}  R={r['instance']['recall']:.4f}")
    else:
        print(f'{name:8s}  no ground truth')

=== Summary (pixel / instance) ===
File1     pixel F1=0.2997  P=0.2132  R=0.5046   |   instance F1=0.2667  P=1.0000  R=0.1538
File2     pixel F1=0.0537  P=0.0294  R=0.3133   |   instance F1=0.3030  P=0.8333  R=0.1852
File3     pixel F1=0.0949  P=0.0531  R=0.4424   |   instance F1=0.4800  P=0.8571  R=0.3333
File4     pixel F1=0.1514  P=0.0956  R=0.3636   |   instance F1=0.3333  P=1.0000  R=0.2000
File5     pixel F1=0.0073  P=0.0037  R=0.5333   |   instance F1=0.5000  P=0.4000  R=0.6667
